# Dwellsy Preliminary Models

Train and compare preliminary rent prediction models on the prepared Chicago Dwellsy dataset.

**Data:** real (Dwellsy)

**Inputs:** chicago_prepared_full.csv

**Outputs:** model metrics and plots

**Run after:** 02_data_preparation.ipynb

In [1]:
seed = 710

In [2]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.preprocessing import OrdinalEncoder


ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "libs").is_dir())
sys.path.insert(0, str(ROOT))   # lets the notebook import the shared libs/ package

from libs import paths
from libs.real_data import raw_file


# 10. Preliminary Rental Price Prediction Models

## 1. Introduction

This notebook develops and evaluates preliminary machine learning models for rental price prediction using historical rental listings from the Chicago Metropolitan Statistical Area (MSA).

The modeling dataset was prepared in `09_dwellsy_data_preparation.ipynb` and contains 545,312 listing records from 2020 to 2026.

The objectives are to:

- Establish a baseline for rental price prediction.
- Train and compare preliminary regression models.
- Evaluate predictive performance using MAE, RMSE, and $R^2$.
- Use group-aware training, validation, and test splits to prevent the same rental unit from appearing in multiple datasets.
- Compare preliminary models using validation performance while reserving the test set for final evaluation.
- Identify promising approaches for subsequent model development.

Missing value imputation and categorical encoding will be performed within preprocessing pipelines fitted exclusively on the training data to reduce the risk of data leakage.

The results will serve as preliminary benchmarks rather than final optimized models.


## 2. Data Loading and Feature Selection

### 2.1 Loading the Prepared Dataset

We load the complete Chicago dataset generated in `09_dwellsy_data_preparation.ipynb`. The dataset contains rental prices, property characteristics, geographic coordinates, temporal features, and rental unit identifiers.

The prepared dataset incorporates data quality corrections, including the treatment of implausible values in `square_feet` and `full_baths`.

Rental unit identifiers will be used to construct group-aware training, validation, and test sets, preventing observations from the same rental unit from appearing in multiple partitions.


In [3]:
df = pd.read_csv(
    paths.PROCESSED / "chicago_prepared_full.csv",
    dtype={"address_zip": "string"},
    parse_dates=["creation_time"],
    low_memory=False
)

# Verify dataset dimensions
print("Dataset shape:", df.shape)

print("\nAvailable columns:")
print(df.columns.tolist())

# Check geographic variables
print("\nGeographic coordinate availability:")
print(df[["latitude", "longitude"]].notna().sum())

display(df.head())


Dataset shape: (545312, 49)

Available columns:
['dwellsy_address_id', 'address1_id', 'address2_id', 'data_source_external_id', 'listing_id', 'community_id', 'community_name', 'company_name', 'top_down_community_count', 'top_down_address1_count', 'address_1', 'address_2', 'address_city', 'address_state', 'address_zip', 'address_country', 'address_type', 'latitude', 'longitude', 'msa_code', 'msa_name', 'year_built', 'bedrooms', 'full_baths', 'half_baths', 'square_feet', 'rent_amount', 'listing_deposit', 'property_listing_status', 'description', 'creation_time', 'deactivation_time', 'ss_raw_primary_number', 'ss_raw_street_predirection', 'ss_raw_street_name', 'ss_raw_street_suffix', 'ss_raw_street_postdirection', 'ss_raw_secondary_designator', 'ss_raw_secondary_number', 'ss_raw_default_city_name', 'ss_raw_state_abbreviation', 'ss_raw_zipcode', 'ss_raw_plus4_code', 'amenities', 'photos', 'uru_id', 'uru_type', 'listing_year', 'listing_month']

Geographic coordinate availability:
latitude   

,dwellsy_address_id,address1_id,address2_id,data_source_external_id,listing_id,community_id,community_name,company_name,top_down_community_count,top_down_address1_count,...,ss_raw_default_city_name,ss_raw_state_abbreviation,ss_raw_zipcode,ss_raw_plus4_code,amenities,photos,uru_id,uru_type,listing_year,listing_month
0,133822-8163780,133822,8163780.0,1739819,3675156,279577,6211 N Kenmore Ave,Mihail Marinov,28,28,...,Chicago,IL,60660,2151,12 Month Lease or Longer; Cable Ready; Cats OK...,https://media.dwellsy.com/prod/c151dd5f3919432...,542230,Apartment,2021,2
1,129521-8112041,129521,8112041.0,3871623,3558998,6069979,Mode Logan Square,Jessica Goodnight,296,218,...,Chicago,IL,60642,6024,12 Month Lease or Longer; Air Conditioning; Ca...,https://media.dwellsy.com/prod/8c186d97caa2102...,1362919,Apartment,2021,2
2,123813-8066896,123813,8066896.0,14582015,16668829,279329,21st Street Lofts,3L Living,137,137,...,Chicago,IL,60616,1732,Bike Storage; Cats OK; Deck/Porch/Patio; Eleva...,https://media.dwellsy.com/prod/9f1aec6e910f4f3...,5009610,Apartment,2025,5
3,123813-8066896,123813,8066896.0,14582015,19878436,279329,21st Street Lofts,3L Living,137,137,...,Chicago,IL,60616,1732,Bike Storage; Cats OK; Deck/Porch/Patio; Eleva...,https://media.dwellsy.com/prod/9f1aec6e910f4f3...,5009610,Apartment,2026,5
4,130630-8121250,130630,8121250.0,10139825,10639335,5764,2420 N Kedzie Blvd,Sam,54,54,...,Chicago,IL,60647,5740,Cats OK; Dishwasher; Gas included in rent; Har...,https://media.dwellsy.com/prod/cd5a80b0a1d4a17...,428523,Apartment,2023,6


### 2.2 Defining Features, Target, and Groups

The target variable is `rent_amount`, representing the advertised monthly rental price.

The preliminary models use eight predictors covering property characteristics, location, property type, and temporal information.

Although the complete dataset includes geographic coordinates, latitude and longitude are excluded from these initial models. They will be considered in subsequent geospatial modeling experiments.

Listing identifiers are excluded from the predictor set. However, `uru_id` is retained separately as a grouping variable to ensure that historical records belonging to the same rental unit are assigned to only one of the training, validation, or test sets.


In [4]:
# Define predictor variables
feature_columns = [
    "bedrooms",
    "full_baths",
    "square_feet",
    "year_built",
    "address_zip",
    "uru_type",
    "listing_year",
    "listing_month"
]

# Define features, target, and groups
X = df[feature_columns].copy()
y = df["rent_amount"].copy()
groups = df["uru_id"].copy()

print("Features shape:", X.shape)
print("Target shape:", y.shape)
print("Number of unique rental units:", groups.nunique())

print("\nPredictor variables:")
print(X.columns.tolist())

display(X.head())


Features shape: (545312, 8)
Target shape: (545312,)
Number of unique rental units: 152233

Predictor variables:
['bedrooms', 'full_baths', 'square_feet', 'year_built', 'address_zip', 'uru_type', 'listing_year', 'listing_month']


,bedrooms,full_baths,square_feet,year_built,address_zip,uru_type,listing_year,listing_month
0,1,1.0,800.0,NaN,60660,Apartment,2021,2
1,1,1.0,909.0,NaN,60642,Apartment,2021,2
2,1,1.0,572.0,2017.0,60616,Apartment,2025,5
3,1,1.0,572.0,2017.0,60616,Apartment,2026,5
4,0,1.0,400.0,1929.0,60647,Apartment,2023,6


## 3. Group-Aware Train-Validation-Test Split

The dataset is divided into three mutually exclusive subsets using `GroupShuffleSplit` from scikit-learn, with `uru_id` as the grouping variable.

Because the dataset contains historical observations of the same rental units, a random observation-level split could place records from the same unit in multiple subsets, potentially producing overly optimistic performance estimates.

To reduce this risk, all observations belonging to a given rental unit are assigned to the same subset.

The data are divided into approximately:

- **64% training:** used to fit preprocessing transformations and model parameters.
- **16% validation:** used to compare preliminary models and guide model selection.
- **20% test:** reserved for final model evaluation.

The splitting procedure uses a fixed random seed of 710 for reproducibility. The resulting subsets are checked to ensure that no rental unit appears in more than one subset.


In [5]:

# First split: development and test sets
test_splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=seed
)

dev_idx, test_idx = next(
    test_splitter.split(X, y, groups=groups)
)

X_dev = X.iloc[dev_idx].copy()
X_test = X.iloc[test_idx].copy()

y_dev = y.iloc[dev_idx].copy()
y_test = y.iloc[test_idx].copy()

groups_dev = groups.iloc[dev_idx].copy()
groups_test = groups.iloc[test_idx].copy()

# Second split: training and validation sets
val_splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=seed
)

train_idx, val_idx = next(
    val_splitter.split(X_dev, y_dev, groups=groups_dev)
)

X_train = X_dev.iloc[train_idx].copy()
X_val = X_dev.iloc[val_idx].copy()

y_train = y_dev.iloc[train_idx].copy()
y_val = y_dev.iloc[val_idx].copy()

groups_train = groups_dev.iloc[train_idx].copy()
groups_val = groups_dev.iloc[val_idx].copy()

# Verify that rental units do not overlap
train_units = set(groups_train)
val_units = set(groups_val)
test_units = set(groups_test)

assert train_units.isdisjoint(val_units)
assert train_units.isdisjoint(test_units)
assert val_units.isdisjoint(test_units)

# Display split summary
print("Training observations:", len(X_train))
print("Validation observations:", len(X_val))
print("Test observations:", len(X_test))

print("\nTraining units:", len(train_units))
print("Validation units:", len(val_units))
print("Test units:", len(test_units))

print("\nTraining proportion:", round(len(X_train) / len(X), 4))
print("Validation proportion:", round(len(X_val) / len(X), 4))
print("Test proportion:", round(len(X_test) / len(X), 4))

print("\nNo overlapping rental units across splits.")


Training observations: 348494
Validation observations: 87942
Test observations: 108876

Training units: 97428
Validation units: 24358
Test units: 30447

Training proportion: 0.6391
Validation proportion: 0.1613
Test proportion: 0.1997

No overlapping rental units across splits.


## 4. Data Preprocessing

### 4.1 Preprocessing Pipeline

The dataset contains numerical and categorical predictors that require different preprocessing procedures.

**Numerical features:**
- Missing values are imputed using the median of each feature in the training set.
- Numerical variables are not standardized for these preliminary models.

**Categorical features:**
- Missing values are imputed using the most frequent category in the training set.
- Categorical variables are transformed using one-hot encoding.
- Previously unseen categories are handled using `handle_unknown="ignore"`.

Preprocessing is implemented using scikit-learn's `Pipeline` and `ColumnTransformer`.

For each model, preprocessing transformations are fitted exclusively on the training set. The fitted transformations are then applied to the validation set without recalculating the imputation statistics or category mappings.

This approach reduces the risk of data leakage and ensures consistent preprocessing across training and validation data.

The preprocessing pipeline described above is used for Linear Regression, Decision Tree, and Random Forest. Histogram-based Gradient Boosting uses a separate preprocessing pipeline with ordinal encoding and native categorical feature handling, as described in Section 9. All preprocessing transformations are fitted exclusively on the training set.


In [6]:
# Define numerical and categorical features
numeric_features = [
    "bedrooms",
    "full_baths",
    "square_feet",
    "year_built",
    "listing_year",
    "listing_month"
]

categorical_features = [
    "address_zip",
    "uru_type"
]

# Numerical preprocessing
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

# Categorical preprocessing
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

# Combine preprocessing steps
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

print("Numerical features:", numeric_features)
print("Categorical features:", categorical_features)
print("\nPreprocessing pipeline created successfully.")


Numerical features: ['bedrooms', 'full_baths', 'square_feet', 'year_built', 'listing_year', 'listing_month']
Categorical features: ['address_zip', 'uru_type']

Preprocessing pipeline created successfully.


## 5. Baseline Model

### 5.1 Dummy Regressor

A baseline model is established using `DummyRegressor` with the mean strategy.

This model predicts the average rental price observed in the training set for every observation, regardless of property characteristics.

The baseline provides a reference against which more sophisticated regression models can be evaluated.

Model performance is assessed on the validation set using:

- **Mean Absolute Error (MAE):** Average absolute difference between actual and predicted rental prices.
- **Root Mean Squared Error (RMSE):** Square root of the average squared prediction error, which penalizes larger errors more heavily.
- **$R^2$:** Proportion of variation in validation rental prices explained by the model relative to predicting the validation-set mean.

Lower MAE and RMSE values indicate better predictive performance, while higher $R^2$ values indicate better model fit. Negative $R^2$ values indicate performance worse than predicting the validation-set mean.


In [7]:
# Initialize and train the baseline model
dummy_model = DummyRegressor(strategy="mean")
dummy_model.fit(X_train, y_train)

# Generate validation predictions
y_pred_dummy = dummy_model.predict(X_val)

# Evaluate baseline performance on the validation set
dummy_mae = mean_absolute_error(y_val, y_pred_dummy)
dummy_rmse = np.sqrt(mean_squared_error(y_val, y_pred_dummy))
dummy_r2 = r2_score(y_val, y_pred_dummy)

print(f"Training mean rent: ${y_train.mean():,.2f}")
print("\nBaseline Model Validation Performance:")
print(f"MAE:  ${dummy_mae:,.2f}")
print(f"RMSE: ${dummy_rmse:,.2f}")
print(f"R\u00b2:   {dummy_r2:.4f}")


Training mean rent: $1,917.31

Baseline Model Validation Performance:
MAE:  $668.83
RMSE: $969.97
R²:   -0.0002


### 5.2 Baseline Results

The baseline model predicts a constant monthly rent of $1,917.31, corresponding to the mean rental price in the training set.

On the validation set, the model achieves:

- **MAE:** $668.83
- **RMSE:** $969.97
- **$R^2$:** -0.0002

These results indicate that predicting the training-set mean produces substantial prediction errors and explains virtually none of the variation in validation rental prices.

The slightly negative $R^2$ indicates that the model performs marginally worse than predicting the mean rental price of the validation set.

The baseline provides a benchmark for evaluating whether models incorporating property characteristics, location, and temporal information can improve predictive performance.


## 6. Linear Regression

### 6.1 Model Training and Evaluation

Linear regression is used as the first predictive model incorporating property characteristics, location, property type, and temporal features.

The model is implemented using a scikit-learn pipeline that combines numerical and categorical preprocessing with ordinary least squares regression.

The model is fitted on the training set and evaluated on the validation set using MAE, RMSE, and $R^2$.

Linear regression provides an interpretable benchmark for comparison with more flexible nonlinear models. However, its predictions may be sensitive to extreme feature values and violations of the assumed linear relationship.


In [8]:
# Create linear regression pipeline
linear_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("regressor", LinearRegression())
])

# Train model
linear_model.fit(X_train, y_train)

# Generate validation predictions
y_pred_linear = linear_model.predict(X_val)

# Evaluate performance on the validation set
linear_mae = mean_absolute_error(y_val, y_pred_linear)
linear_rmse = np.sqrt(mean_squared_error(y_val, y_pred_linear))
linear_r2 = r2_score(y_val, y_pred_linear)

print("Linear Regression Validation Performance:")
print(f"MAE:  ${linear_mae:,.2f}")
print(f"RMSE: ${linear_rmse:,.2f}")
print(f"R\u00b2:   {linear_r2:.4f}")


Linear Regression Validation Performance:
MAE:  $334.62
RMSE: $551.74
R²:   0.6764


In [9]:
# Examine prediction errors on the validation set
linear_errors = pd.DataFrame({
    "actual_rent": y_val.to_numpy(),
    "predicted_rent": y_pred_linear
})

linear_errors["absolute_error"] = (
    linear_errors["actual_rent"] -
    linear_errors["predicted_rent"]
).abs()

# Summarize predictions and errors
print("Prediction and error summary:")
display(
    linear_errors[
        ["actual_rent", "predicted_rent", "absolute_error"]
    ].describe(percentiles=[0.50, 0.90, 0.95, 0.99, 0.999])
)

# Identify the largest prediction errors
print("\nTop 10 largest prediction errors:")
display(
    linear_errors.nlargest(10, "absolute_error")
)


Prediction and error summary:


,actual_rent,predicted_rent,absolute_error
count,87942.000000,87942.000000,87942.000000
mean,1904.856515,1911.716957,334.619517
std,969.894932,816.013439,438.683880
min,300.000000,-137.100858,0.011117
50%,1677.000000,1743.543319,231.039549
90%,2995.000000,2978.827174,698.935392
95%,3592.950000,3542.785679,914.857925
99%,5350.000000,4238.424221,1840.453947
99.9%,9554.000000,5299.293896,4696.107818
max,20000.000000,7554.047572,16017.176333



Top 10 largest prediction errors:


,actual_rent,predicted_rent,absolute_error
33651,20000.0,3982.823667,16017.176333
66060,20000.0,3982.823667,16017.176333
25233,19500.0,4434.371156,15065.628844
75103,19500.0,4434.371156,15065.628844
25234,19500.0,4456.696092,15043.303908
78426,20000.0,5754.089555,14245.910445
78427,20000.0,5765.252023,14234.747977
78428,20000.0,5765.252023,14234.747977
78429,20000.0,5776.414491,14223.585509
78430,20000.0,5787.576959,14212.423041


### 6.2 Linear Regression Results

The linear regression model achieved the following performance on the validation set:

- **MAE:** $334.62
- **RMSE:** $551.74
- **$R^2$:** 0.6764

Compared with the baseline model, linear regression reduces the MAE from $668.83 to $334.62, an improvement of approximately 50%.

The model explains approximately 67.64% of the variation in rental prices in the validation set.

An examination of prediction errors reveals that the model tends to substantially underestimate some high-priced listings. The largest absolute error is approximately $16,017, corresponding to an actual monthly rent of $20,000 and a predicted rent of approximately $3,983.

The model also produces a small number of negative rental price predictions, reflecting the absence of non-negativity constraints in ordinary least squares regression.

Overall, linear regression provides a useful and interpretable preliminary benchmark. However, its limitations suggest that nonlinear models may better capture complex relationships between property characteristics and rental prices.


In [10]:
# Compare preliminary model performance on the validation set
results_df = pd.DataFrame({
    "Model": ["Dummy Regressor", "Linear Regression"],
    "MAE": [dummy_mae, linear_mae],
    "RMSE": [dummy_rmse, linear_rmse],
    "R\u00b2": [dummy_r2, linear_r2]
})

results_df = results_df.sort_values("MAE").reset_index(drop=True)

display(
    results_df.style.format({
        "MAE": "${:,.2f}",
        "RMSE": "${:,.2f}",
        "R\u00b2": "{:.4f}"
    })
)


,Model,MAE,RMSE,R²
0,Linear Regression,$334.62,$551.74,0.6764
1,Dummy Regressor,$668.83,$969.97,-0.0002


## 7. Decision Tree Regression

### 7.1 Model Training and Evaluation

A Decision Tree Regressor is trained to capture nonlinear relationships and interactions among property characteristics, location, and temporal features.

Unlike linear regression, decision trees do not assume a linear relationship between predictors and rental prices.

The model is implemented using the same preprocessing pipeline and group-aware train-validation-test split as the previous models. Training is performed exclusively on the training set, while model performance is assessed on the validation set.

A maximum tree depth of 10 is used as an initial complexity constraint to reduce overfitting. This value is not optimized at this stage; hyperparameter tuning is deferred to subsequent experiments.

Performance is evaluated on the validation set using MAE, RMSE, and $R^2$.


In [11]:
# Create decision tree pipeline
tree_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("regressor", DecisionTreeRegressor(
        max_depth=10,
        random_state=seed
    ))
])

# Train model
tree_model.fit(X_train, y_train)

# Generate predictions on the validation set
y_pred_tree = tree_model.predict(X_val)

# Evaluate validation performance
tree_mae = mean_absolute_error(y_val, y_pred_tree)
tree_rmse = np.sqrt(mean_squared_error(y_val, y_pred_tree))
tree_r2 = r2_score(y_val, y_pred_tree)

print("Decision Tree Regression Validation Performance:")
print(f"MAE:  ${tree_mae:,.2f}")
print(f"RMSE: ${tree_rmse:,.2f}")
print(f"R²:   {tree_r2:.4f}")


Decision Tree Regression Validation Performance:
MAE:  $345.79
RMSE: $533.11
R²:   0.6979


### 7.2 Decision Tree Results

The Decision Tree Regressor achieves the following performance on the validation set:

- **MAE:** $345.79
- **RMSE:** $533.11
- **$R^2$:** 0.6979

Compared with linear regression, the decision tree produces a slightly higher MAE (approximately 3.3%) but reduces RMSE by approximately 3.4%.

The improvement in RMSE and $R^2$ suggests that the decision tree may capture nonlinear relationships and interactions that are not fully represented by the linear model.

However, the higher MAE indicates that the decision tree does not outperform linear regression across all evaluation metrics.

These findings motivate further experimentation with ensemble methods, such as Random Forest and Gradient Boosting.


In [12]:
# Add Decision Tree results
results_df = pd.concat([
    results_df,
    pd.DataFrame({
        "Model": ["Decision Tree"],
        "MAE": [tree_mae],
        "RMSE": [tree_rmse],
        "R\u00b2": [tree_r2]
    })
], ignore_index=True)

# Sort models by MAE
results_df = results_df.sort_values("MAE").reset_index(drop=True)

# Display formatted results
display(
    results_df.style.format({
        "MAE": "${:,.2f}",
        "RMSE": "${:,.2f}",
        "R\u00b2": "{:.4f}"
    })
)


,Model,MAE,RMSE,R²
0,Linear Regression,$334.62,$551.74,0.6764
1,Decision Tree,$345.79,$533.11,0.6979
2,Dummy Regressor,$668.83,$969.97,-0.0002


## 8. Random Forest Regression

### 8.1 Model Training and Evaluation

Random Forest is an ensemble learning method that combines predictions from multiple decision trees trained on bootstrap samples of the training data.

By averaging predictions across trees, Random Forest can reduce prediction variance and improve generalization compared with a single decision tree.

For this preliminary experiment, the model uses 100 trees, a maximum depth of 15, and a minimum of five samples per leaf. These hyperparameters are selected as initial settings rather than through systematic optimization.

The same preprocessing pipeline and group-aware train-validation-test split are used to ensure comparability with previous models. Training is performed exclusively on the training set, while model performance is assessed on the validation set.

Performance is evaluated on the validation set using MAE, RMSE, and $R^2$.


In [13]:
# Create Random Forest pipeline
rf_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("regressor", RandomForestRegressor(
        n_estimators=100,
        max_depth=15,
        min_samples_leaf=5,
        random_state=seed,
        n_jobs=-1
    ))
])

# Train model
rf_model.fit(X_train, y_train)

# Generate predictions on the validation set
y_pred_rf = rf_model.predict(X_val)

# Evaluate validation performance
rf_mae = mean_absolute_error(y_val, y_pred_rf)
rf_rmse = np.sqrt(mean_squared_error(y_val, y_pred_rf))
rf_r2 = r2_score(y_val, y_pred_rf)

print("Random Forest Regression Validation Performance:")
print(f"MAE:  ${rf_mae:,.2f}")
print(f"RMSE: ${rf_rmse:,.2f}")
print(f"R²:   {rf_r2:.4f}")


Random Forest Regression Validation Performance:
MAE:  $291.17
RMSE: $464.58
R²:   0.7706


### 8.2 Random Forest Results

The Random Forest Regressor achieves the following performance on the validation set:

- **MAE:** $291.17
- **RMSE:** $464.58
- **$R^2$:** 0.7706

Random Forest outperforms all previously evaluated models across the three performance metrics.

Compared with linear regression, Random Forest reduces MAE by approximately 13.0% and RMSE by 15.8%.

The model explains approximately 77.1% of the variation in rental prices in the validation set.

These findings suggest that ensemble methods can capture nonlinear relationships and interactions among property characteristics, geographic information represented by ZIP codes, and temporal features more effectively than the preliminary linear regression and individual decision tree models.

Further evaluation and hyperparameter tuning are needed before selecting a final model.


In [14]:
# Add Random Forest results
results_df = pd.concat([
    results_df,
    pd.DataFrame({
        "Model": ["Random Forest"],
        "MAE": [rf_mae],
        "RMSE": [rf_rmse],
        "R\u00b2": [rf_r2]
    })
], ignore_index=True)

# Sort models by MAE
results_df = results_df.sort_values("MAE").reset_index(drop=True)

# Display formatted results
display(
    results_df.style.format({
        "MAE": "${:,.2f}",
        "RMSE": "${:,.2f}",
        "R\u00b2": "{:.4f}"
    })
)


,Model,MAE,RMSE,R²
0,Random Forest,$291.17,$464.58,0.7706
1,Linear Regression,$334.62,$551.74,0.6764
2,Decision Tree,$345.79,$533.11,0.6979
3,Dummy Regressor,$668.83,$969.97,-0.0002


## 9. Gradient Boosting Regression

### 9.1 Model Training and Evaluation

Gradient Boosting is an ensemble learning method that builds decision trees sequentially, with each new tree improving the predictions of the existing ensemble by reducing a loss function.

Unlike Random Forest, which averages predictions from multiple independently trained trees, Gradient Boosting iteratively updates the ensemble to minimize prediction errors.

We use `HistGradientBoostingRegressor`, an efficient histogram-based implementation designed for large datasets.

The model uses the same input features and group-aware train-validation-test split as the previous models. Training is performed exclusively on the training set, while performance is assessed on the validation set.

Performance is evaluated on the validation set using MAE, RMSE, and $R^2$.


In [15]:
# Define features
numerical_features = [
    "bedrooms",
    "full_baths",
    "square_feet",
    "year_built",
    "listing_year",
    "listing_month"
]

categorical_features = [
    "address_zip",
    "uru_type"
]

# Group infrequent categories using training data only
class FrequentCategoryGrouper(BaseEstimator, TransformerMixin):
    def __init__(self, max_categories=200):
        self.max_categories = max_categories

    def fit(self, X, y=None):
        X = pd.DataFrame(X)
        self.frequent_categories_ = []

        for col in X.columns:
            frequent = (
                X[col]
                .value_counts()
                .head(self.max_categories - 1)
                .index
            )
            self.frequent_categories_.append(set(frequent))

        return self

    def transform(self, X):
        X = pd.DataFrame(X).copy()

        for i, col in enumerate(X.columns):
            X[col] = (
                X[col]
                .where(
                    X[col].isin(self.frequent_categories_[i]),
                    "OTHER"
                )
            )

        return X

# Preprocessing for Gradient Boosting
gb_preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            SimpleImputer(strategy="median"),
            numerical_features
        ),
        (
            "cat",
            Pipeline(steps=[
                (
                    "imputer",
                    SimpleImputer(strategy="most_frequent")
                ),
                (
                    "grouper",
                    FrequentCategoryGrouper(max_categories=200)
                ),
                (
                    "encoder",
                    OrdinalEncoder(
                        handle_unknown="use_encoded_value",
                        unknown_value=-1
                    )
                )
            ]),
            categorical_features
        )
    ]
)

# Create Gradient Boosting pipeline
gb_model = Pipeline(steps=[
    ("preprocessor", gb_preprocessor),
    ("regressor", HistGradientBoostingRegressor(
        max_iter=200,
        learning_rate=0.1,
        max_leaf_nodes=31,
        categorical_features=[6, 7],
        random_state=seed
    ))
])

# Train model
gb_model.fit(X_train, y_train)

# Generate predictions on the validation set
y_pred_gb = gb_model.predict(X_val)

# Evaluate validation performance
gb_mae = mean_absolute_error(y_val, y_pred_gb)
gb_rmse = np.sqrt(mean_squared_error(y_val, y_pred_gb))
gb_r2 = r2_score(y_val, y_pred_gb)

print("Gradient Boosting Regression Validation Performance:")
print(f"MAE:  ${gb_mae:,.2f}")
print(f"RMSE: ${gb_rmse:,.2f}")
print(f"R²:   {gb_r2:.4f}")


Gradient Boosting Regression Validation Performance:
MAE:  $229.22
RMSE: $396.24
R²:   0.8331


### 9.2 Gradient Boosting Results

The HistGradientBoostingRegressor achieves the following performance on the validation set:

- **MAE:** $229.22
- **RMSE:** $396.24
- **$R^2$:** 0.8331

Gradient Boosting achieves the best predictive performance among all preliminary models evaluated.

Compared with Random Forest, it reduces MAE by approximately 21.3% and RMSE by 14.7%.

The model explains approximately 83.3% of the variation in rental prices in the validation set.

For this model, infrequent categorical values were grouped to accommodate the maximum categorical cardinality supported by HistGradientBoostingRegressor. Categorical variables were also ordinal-encoded and explicitly identified as categorical features in the model.

These results suggest that Gradient Boosting is a promising candidate for further evaluation and hyperparameter tuning. Final performance should be assessed on the reserved test set after model selection.


In [16]:
# Add Gradient Boosting results
results_df = pd.concat([
    results_df,
    pd.DataFrame({
        "Model": ["Gradient Boosting"],
        "MAE": [gb_mae],
        "RMSE": [gb_rmse],
        "R\u00b2": [gb_r2]
    })
], ignore_index=True)

# Sort models by MAE
results_df = results_df.sort_values("MAE").reset_index(drop=True)

# Display formatted results
display(
    results_df.style.format({
        "MAE": "${:,.2f}",
        "RMSE": "${:,.2f}",
        "R\u00b2": "{:.4f}"
    })
)


,Model,MAE,RMSE,R²
0,Gradient Boosting,$229.22,$396.24,0.8331
1,Random Forest,$291.17,$464.58,0.7706
2,Linear Regression,$334.62,$551.74,0.6764
3,Decision Tree,$345.79,$533.11,0.6979
4,Dummy Regressor,$668.83,$969.97,-0.0002


## 10. Preliminary Model Comparison and Next Steps

Five regression models were evaluated to predict rental prices in Chicago: Dummy Regressor, Linear Regression, Decision Tree, Random Forest, and Histogram-based Gradient Boosting.

Gradient Boosting achieved the best preliminary predictive performance on the validation set, with an MAE of $229.22, RMSE of $396.24, and $R^2$ of 0.8331. Compared with Random Forest, it reduced MAE by approximately 21.3% and RMSE by 14.7%.

All models were evaluated using the same group-aware data split, with rental units identified by `uru_id`. This ensures that observations from the same rental unit do not appear across the training, validation, and test sets.

However, predictive accuracy is not the only criterion for model selection. Interpretability is an important requirement for Accolade, and geographic variation is particularly relevant to rental price estimation.

The next phase will therefore focus on:

1. Evaluating the availability and quality of geographic variables, including latitude and longitude.
2. Developing an interpretable geospatial regression model to capture geographic variation in rental prices.
3. Comparing predictive performance and interpretability across models.
4. Exploring feature engineering and hyperparameter tuning to improve predictive performance.
5. Maintaining the group-aware validation strategy and reserving the test set for final model evaluation.

These preliminary results establish a benchmark for subsequent modeling experiments and identify Gradient Boosting as a promising candidate for further development.


In [17]:
# Identify geographic variables in the dataset
geo_keywords = [
    "lat", "lon", "lng", "coord",
    "zip", "city", "state",
    "county", "neighborhood", "address"
]

geo_columns = [
    col for col in df.columns
    if any(keyword in col.lower() for keyword in geo_keywords)
]

print(f"Number of geographic variables identified: {len(geo_columns)}")

print("\nGeographic variables available in the dataset:")
for col in geo_columns:
    print(f"- {col}")


Number of geographic variables identified: 16

Geographic variables available in the dataset:
- dwellsy_address_id
- address1_id
- address2_id
- top_down_address1_count
- address_1
- address_2
- address_city
- address_state
- address_zip
- address_country
- address_type
- latitude
- longitude
- ss_raw_default_city_name
- ss_raw_state_abbreviation
- ss_raw_zipcode
